In [ ]:
# ============================================================
# ADES (Adaptive Dynamic Epsilon Scheduling) — MALEVIS
# Replaces fixed epsilon_q in QNI-CCP training with a
# per-sample adaptive epsilon computed from three cues:
#   1. Gradient norm   — sensitivity to small feature changes
#   2. Confidence entropy — model confusion on this sample
#   3. MC Dropout variance — model uncertainty (T=10 passes)
# Formula: epsilon_x = epsilon_min + lambda * sigma(x)
# sigma(x) = equal-weight average of all three normalized cues
#
# Architecture: identical to your Malevis QNI-CCP training.
# Training loop: identical structure — only perturbation
# function changes. Everything else stays the same.
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import numpy as np
import random
import os
import math
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm.notebook import tqdm
from sklearn.utils.class_weight import compute_class_weight

# ========== SEEDING ==========
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== CONFIGURATION ==========
n_qubits    = 10
num_classes = 26
batch_size  = 16
num_epochs  = 70
lr          = 0.0003

# ── ADES Hyperparameters ──────────────────────────────────────
# epsilon_min: baseline robustness floor — every sample gets
# at least this much perturbation regardless of confidence.
# Equivalent to epsilon_q from fixed training (≈0.0164).
EPSILON_MIN  = 0.0164   # 1/(1 + 1.0×54 + 1.0×6) — Malevis circuit

# lambda_max: maximum dynamic range on top of epsilon_min.
# Total epsilon per sample: epsilon_min + lambda * sigma(x)
# sigma(x) in [0,1], so max total epsilon = epsilon_min + lambda_max
# Set based on your sweep analysis: Malevis showed meaningful
# robustness gap forming at E=2.5-3.0, so lambda_max=2.0 gives
# range [0.0164, 2.0164] — covers full useful training range.
LAMBDA_MAX   = 3      # Max dynamic epsilon range above floor

# T: number of MC Dropout forward passes for uncertainty esti"mation
# Higher T = more accurate variance but T× slower per batch
MC_T         = 10       # Balanced accuracy/speed tradeoff

# Checkpoint paths
BASIC_CHECKPOINT = "/home/netsec1/Kathan/MaleVis experiments/exp-4_1_3.pth"          # Trained Malevis basic model
ADES_SAVE_PATH   = "qni_ccp_malevis2.pth"  # Where ADES model saves

# ========== TRANSFORMS ==========
train_transform = transforms.Compose([
    transforms.Grayscale(1),
   
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ========== DATASETS ==========
TRAIN_PATH = '/home/netsec1/Kathan/MaleVis experiments/malevis_train_val_300x300/train'
VAL_PATH   = '/home/netsec1/Kathan/MaleVis experiments/malevis_train_val_300x300/val'
TEST_PATH  = '/home/netsec1/Kathan/MaleVis experiments/malevis_train_val_300x300/test'

train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
val_dataset   = ImageFolder(VAL_PATH,   transform=eval_transform)
test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ========== CLASS WEIGHTS ==========
labels_list          = [label for _, label in train_dataset.samples]
class_weights        = compute_class_weight('balanced',
                                             classes=np.unique(labels_list),
                                             y=labels_list)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights computed.")

train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                          shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size,
                          shuffle=False, num_workers=4)

# ========== QUANTUM CIRCUIT ==========
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)     # Angle embedding per qubit
    for l in range(weights.shape[0]):        # 6 variational layers
        for i in range(n_qubits):
            qml.RY(weights[l][i], wires=i)  # Trainable rotation
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])        # Linear entanglement
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (6, n_qubits)}

# ========== MODEL ARCHITECTURE ==========
class FeatureReduce(nn.Module):
    """
    CNN: 224×224 → 10-dim latent vector.
    Two FC layers (fc_expand + fc_project) — must match checkpoint.
    Dropout layers in CNN encoder are CRITICAL for MC Dropout:
    they remain active during model.train() mode, enabling
    stochastic forward passes for uncertainty estimation.
    """
    def __init__(self, final_dim, dropout=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1),
            nn.BatchNorm2d(8), nn.ReLU(), nn.Dropout(dropout),

            nn.Conv2d(8, 16, 3, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(), nn.Dropout(dropout),

            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.Dropout(dropout),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.Dropout(dropout),

            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.Conv2d(128, 224, 3, stride=2, padding=1),
            nn.BatchNorm2d(224), nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc_expand  = nn.Linear(224, final_dim * 2)
        self.fc_project = nn.Linear(final_dim * 2, final_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc_expand(x))
        return self.fc_project(x)


class HybridQNN(nn.Module):
    """
    Hybrid model — identical to APCS training architecture.
    Dropout in both CNN (dropout=0.1) and classifier (dropout=0.15)
    enables MC Dropout uncertainty estimation by running T forward
    passes with model.train() to keep dropout stochastic.
    """
    def __init__(self, n_qubits, num_classes):
        super().__init__()
        self.feature_extractor = FeatureReduce(final_dim=n_qubits)
        self.q_layer           = TorchLayer(quantum_circuit, weight_shapes)
        self.classifier        = nn.Sequential(
            nn.Linear(n_qubits, 64), nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(64, 32),       nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(32, 16),       nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        x     = self.feature_extractor(x)   # [batch, 10]
        x     = torch.tanh(x)              # Squash to [-1,1]
        q_out = self.q_layer(x)            # [batch, 10]
        return self.classifier(q_out)      # [batch, 26]


# ========== FOCAL LOSS ==========
class FocalLoss(nn.Module):
    """Identical to Malimg APCS training — same class, same gamma schedule."""
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(inputs, targets, reduction='none')
        pt         = torch.exp(-ce_loss)
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()


# ========== ADES CORE FUNCTIONS ==========

def compute_gradient_norm(model, features, labels):
    """
    Cue 1: Gradient norm — measures sensitivity of each sample.

    High gradient norm → the model's loss changes rapidly with
    small feature changes → this sample is near a decision boundary
    → it is sensitive → assign moderate-to-high epsilon.

    Low gradient norm → confident prediction, loss barely changes
    → sample is deep inside its class region.

    Process:
        features (requires_grad=True) → quantum → classifier → loss
        → backward → ||grad||_2 per sample

    Returns:
        grad_norms : Tensor [batch] — L2 norm of gradient per sample.
                     Higher = more sensitive to perturbation.
    """
    # Clone features with gradient tracking
    feats_grad = features.clone().detach().requires_grad_(True)  # [batch, 10]

    # Full forward pass — backprop enabled via diff_method="backprop"
    q_out  = model.q_layer(feats_grad)            # [batch, 10]
    logits = model.classifier(q_out)              # [batch, 26]
    loss   = F.cross_entropy(logits, labels)      # Scalar
    loss.backward()                               # Compute gradients

    if feats_grad.grad is None:
        # Fallback: return zeros if gradient computation failed
        return torch.zeros(features.size(0), device=features.device)

    # L2 norm of gradient vector per sample
    # grad shape: [batch, 10] → norm over feature dim → [batch]
    grad_norms = torch.norm(feats_grad.grad.data, p=2, dim=1)  # [batch]
    return grad_norms


def compute_confidence_entropy(model, features):
    """
    Cue 2: Confidence entropy — measures prediction uncertainty.

    High entropy → model spreads probability across many classes
    → confused about this sample → it is hard → lower epsilon
    (don't overwhelm the model with impossible examples).

    Low entropy → model is very confident about one class
    → easy sample deep in class region → higher epsilon needed
    to push it toward a decision boundary.

    Note: this is the INVERSE of confidence — we want to assign
    higher epsilon to confident (low-entropy) samples because
    they need harder pushing. This directly addresses the Malevis
    flat curve problem caused by overconfident predictions.

    Process:
        features → quantum → classifier → softmax → entropy
        entropy = -sum(p * log(p)) per sample

    Returns:
        entropy : Tensor [batch] — Shannon entropy of softmax output.
                  Higher = more uncertain = assign lower epsilon.
    """
    with torch.no_grad():
        q_out   = model.q_layer(features)          # [batch, 10]
        logits  = model.classifier(q_out)          # [batch, 26]
        probs   = F.softmax(logits, dim=1)         # [batch, 26] probabilities

        # Shannon entropy: H = -sum(p * log(p + eps))
        # eps=1e-8 prevents log(0) numerical instability
        entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1)  # [batch]

    return entropy


def compute_mc_dropout_variance(model, features, T=10):
    """
    Cue 3: MC Dropout variance — Bayesian uncertainty estimation.

    Running T forward passes with dropout ACTIVE (model.train() mode)
    produces T different predictions per sample due to random neuron
    dropout. The variance of these predictions measures how uncertain
    the model is about each sample.

    High variance → model gives very different answers each pass
    → highly uncertain about this sample → assign lower epsilon
    (sample already near boundary, easy to perturb).

    Low variance → model consistently gives same answer each pass
    → very certain → assign higher epsilon (need harder push).

    Why this works differently from entropy:
    Entropy measures single-pass output spread (aleatoric uncertainty).
    MC Dropout variance measures consistency across passes (epistemic
    uncertainty) — a different kind of uncertainty that captures
    whether the model has learned a robust representation.

    Process:
        for t in range(T):
            model.train()  ← keeps dropout active
            logit_t = forward(features)
        variance = var(softmax(logit_t) across T passes) per sample

    Returns:
        mc_variance : Tensor [batch] — mean variance across classes
                      and T passes per sample. Higher = more uncertain.
    """
    # Collect T stochastic predictions
    all_probs = []

    model.train()   # CRITICAL: activates dropout for stochastic passes
    with torch.no_grad():
        for t in range(T):
            q_out  = model.q_layer(features)           # [batch, 10]
            logits = model.classifier(q_out)           # [batch, 26]
            probs  = F.softmax(logits, dim=1)          # [batch, 26]
            all_probs.append(probs.unsqueeze(0))       # [1, batch, 26]

    # Stack all T passes: [T, batch, 26]
    all_probs_stacked = torch.cat(all_probs, dim=0)

    # Variance across T passes for each (sample, class) pair
    # var shape: [batch, 26] → mean over class dim → [batch]
    mc_variance = all_probs_stacked.var(dim=0).mean(dim=1)  # [batch]

    model.eval()    # Restore eval mode after MC passes
    return mc_variance


def normalize_to_01(tensor):
    """
    Min-max normalize a batch tensor to [0, 1] range.

    Required before combining the three cues — they have different
    scales (grad_norm may be 0-50, entropy 0-3.26, variance 0-0.25).
    Without normalization, gradient norm would dominate sigma(x).

    Args:
        tensor : Tensor [batch] — raw cue values
    Returns:
        normalized : Tensor [batch] — values in [0, 1]

    If all values are identical (degenerate batch), returns 0.5
    to avoid division by zero.
    """
    t_min = tensor.min()
    t_max = tensor.max()

    if (t_max - t_min) < 1e-8:
        # Degenerate case: all samples identical → neutral score
        return torch.full_like(tensor, 0.5)

    return (tensor - t_min) / (t_max - t_min)   # [batch] in [0,1]


def compute_ades_epsilon(model, x, labels, epsilon_min, lambda_max, T=10):
    """
    Main ADES function — computes per-sample adaptive epsilon.
    Formula: epsilon_x = epsilon_min + lambda_max * sigma(x)
    sigma(x) = (norm_grad + (1 - norm_entropy) + (1 - norm_variance)) / 3

    Why entropy and variance are INVERTED before averaging:
    - High entropy → uncertain → LOWER epsilon (don't overwhelm)
    - High variance → uncertain → LOWER epsilon (already near boundary)
    - High grad norm → sensitive → HIGHER epsilon (needs hard push)
    So we use (1 - entropy) and (1 - variance) to align all three
    cues in the same direction: higher cue value = higher epsilon.

    This directly addresses the Malevis overconfidence problem:
    confident samples (low entropy, low variance, moderate grad)
    get high sigma → high epsilon → actual perturbation happens.

    Args:
        model       : HybridQNN in eval mode (switched internally)
        x           : Tensor [batch, C, H, W] — raw input images
        labels      : Tensor [batch] — true class labels
        epsilon_min : float — floor epsilon (= quantum circuit epsilon_q)
        lambda_max  : float — max dynamic range above floor
        T           : int — MC Dropout passes for variance estimation

    Returns:
        epsilon_per_sample : Tensor [batch, 1] — per-sample epsilon
                             ready to broadcast over feature dimensions.
        sigma_stats        : dict — monitoring stats for logging
    """
    model.eval()   # Start in eval mode for feature extraction

    # ── Extract clean latent features ─────────────────────────
    with torch.no_grad():
        features = model.feature_extractor(x)   # [batch, 10]
        features = torch.tanh(features)          # [batch, 10] in [-1,1]

    # ── Cue 1: Gradient norm ──────────────────────────────────
    # Re-extract with grad enabled for sensitivity computation
    features_for_grad = features.clone().detach().requires_grad_(True)
    grad_norms        = compute_gradient_norm(model, features_for_grad, labels)
    # grad_norms: [batch] — raw L2 gradient norms

    # ── Cue 2: Confidence entropy ─────────────────────────────
    # Use clean features (no grad needed)
    entropy           = compute_confidence_entropy(model, features)
    # entropy: [batch] — Shannon entropy of softmax output

    # ── Cue 3: MC Dropout variance ────────────────────────────
    mc_variance       = compute_mc_dropout_variance(model, features, T=T)
    # mc_variance: [batch] — variance across T stochastic passes

    # ── Normalize all three cues to [0,1] ─────────────────────
    # Required to prevent scale dominance before averaging
    norm_grad     = normalize_to_01(grad_norms)    # [batch] in [0,1]
    norm_entropy  = normalize_to_01(entropy)       # [batch] in [0,1]
    norm_variance = normalize_to_01(mc_variance)   # [batch] in [0,1]

    # ── Combine into sigma(x) — equal weights ─────────────────
    # Entropy and variance are INVERTED: high uncertainty → lower epsilon
    # Gradient is kept as-is: high sensitivity → higher epsilon
    sigma = (norm_grad + (1.0 - norm_entropy) + (1.0 - norm_variance)) / 3.0
    # sigma: [batch] in [0,1] — normalized difficulty score per sample

    # ── Compute per-sample epsilon ────────────────────────────
    # epsilon_x = epsilon_min + lambda_max * sigma(x)
    # Range: [epsilon_min, epsilon_min + lambda_max]
    epsilon_per_sample = epsilon_min + lambda_max * sigma   # [batch]
    epsilon_per_sample = epsilon_per_sample.unsqueeze(1)    # [batch, 1] for broadcasting

    # ── Stats for monitoring ──────────────────────────────────
    sigma_stats = {
        "eps_mean" : epsilon_per_sample.mean().item(),
        "eps_min"  : epsilon_per_sample.min().item(),
        "eps_max"  : epsilon_per_sample.max().item(),
        "eps_std"  : epsilon_per_sample.std().item(),
        "sigma_mean": sigma.mean().item(),
    }

    return epsilon_per_sample.detach(), sigma_stats


def qni_ccp_ades_perturbation(model, x, y, centroids,
                               epsilon_min, lambda_max, T=10):
    model.eval()

    # ── Step 1: Clean latent features z ───────────────────────
    with torch.no_grad():
        original_features = model.feature_extractor(x)       # (B, 10)
        original_features = torch.tanh(original_features)    # (B, 10)

    # ── Step 2: ADES per-sample epsilon ───────────────────────
    epsilon_per_sample, sigma_stats = compute_ades_epsilon(
        model, x, y, epsilon_min, lambda_max, T=T
    )                                                         # (B, 1)

    # ── Step 3: Sensitivity S via full quantum path, normalized ─
    # diff_method="backprop" allows gradients to flow back through
    # quantum layer to features — use this for accurate sensitivity.
    # Normalize to unit-norm so epsilon_per_sample is pure step size.
    features_for_grad = original_features.clone().detach().requires_grad_(True)
    q_out  = model.q_layer(features_for_grad)   # (B, 10) through quantum
    logits = model.classifier(q_out)            # (B, 26)
    loss   = F.cross_entropy(logits, y)
    loss.backward()

    S_raw = features_for_grad.grad.data if features_for_grad.grad is not None \
            else torch.zeros_like(features_for_grad)          # (B, 10)
    S     = S_raw / (S_raw.norm(dim=1, keepdim=True) + 1e-8) # (B, 10) unit-norm

    # ── Step 4: Random wrong-class centroid selection ──────────
    # This block was accidentally removed — mu_c_prime must be
    # defined before it is used in Step 5.
    target_classes = []
    for i in range(y.size(0)):
        available = [c for c in range(centroids.size(0)) if c != y[i].item()]
        target_classes.append(
            random.choice(available) if available
            else (y[i].item() + 1) % centroids.size(0)
        )
    target_tensor = torch.tensor(target_classes, device=y.device)  # (B,)
    mu_c_prime    = centroids[target_tensor]                        # (B, 10)

    # ── Step 5: Normalized direction + apply perturbation ──────
    # Normalize direction so epsilon is pure step-size regardless
    # of inter-centroid distance — consistent with sweep code.
    direction = mu_c_prime - original_features                      # (B, 10) raw
    dir_hat   = direction / (direction.norm(dim=1, keepdim=True) + 1e-8)  # unit-norm

    weighted  = S * dir_hat                                         # (B, 10)
    perturbed = original_features + epsilon_per_sample * weighted   # (B, 10)

    return perturbed.detach(), sigma_stats


def compute_class_centroids(model, loader, device, num_classes):
    """Identical to APCS training — compute μ_c in latent space."""
    model.eval()
    sums   = torch.zeros(num_classes, n_qubits, device=device)
    counts = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for x, y in loader:
            x, y     = x.to(device), y.to(device)
            features = model.feature_extractor(x)
            features = torch.tanh(features)
            for c in range(num_classes):
                mask = (y == c)
                if mask.any():
                    sums[c]   += features[mask].sum(0)
                    counts[c] += mask.sum()

    counts[counts == 0] = 1
    return sums / counts.unsqueeze(1)   # [26, 10]


def evaluate(model, dataloader, loss_fn, device):
    """Identical to APCS training evaluate()."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs        = model(inputs)
            loss           = loss_fn(outputs, labels)
            total_loss    += loss.item()
            _, predicted   = torch.max(outputs.data, 1)
            total         += labels.size(0)
            correct       += (predicted == labels).sum().item()

    return total_loss / len(dataloader), correct / total


# ========== MAIN EXECUTION ==========

# ── Step 1: Model + load basic checkpoint ────────────────────
print("\nInitializing ADES QNI-CCP Training — Malevis...")
model = HybridQNN(n_qubits=n_qubits, num_classes=num_classes).to(device)

if os.path.exists(BASIC_CHECKPOINT):
    ckpt  = torch.load(BASIC_CHECKPOINT, map_location=device)
    state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and \
            'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state)
    print(f"  Loaded: {BASIC_CHECKPOINT}")
else:
    print(f"  ⚠️  Checkpoint not found — training from scratch.")

# ── Step 2: Compute centroids on loaded model ─────────────────
print("Computing class centroids...")
centroids = compute_class_centroids(model, train_loader, device, num_classes)

# ── Step 3: Optimizer, loss, scheduler ───────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.005)
loss_fn   = FocalLoss(alpha=1, gamma=1.5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5   # track val_acc now
)

best_val_acc              = 0.0
epochs_without_improvement = 0
early_stopping_patience   = 15

print(f"\nStarting ADES QNI-CCP Training for {num_epochs} epochs...")
print(f"  epsilon_min={EPSILON_MIN} | lambda_max={LAMBDA_MAX} | MC_T={MC_T}")
print(f"  Per-sample epsilon range: [{EPSILON_MIN:.4f}, {EPSILON_MIN+LAMBDA_MAX:.4f}]")
print("="*60)

# ========== TRAINING LOOP ==========
for epoch in range(num_epochs):

    # ── Two-stage focal loss — identical to APCS ──────────────
    if epoch < 10:
        loss_fn.gamma = 0.0
    elif epoch < 30:
        loss_fn.gamma = 0.5
    else:
        loss_fn.gamma = 1.5

    # ── Recompute centroids every 5 epochs — same as APCS ─────
    if epoch > 0 and epoch % 5 == 0:
        centroids = compute_class_centroids(model, train_loader, device, num_classes)
        print(f"  Centroids updated at epoch {epoch}")

    # ── Training pass ─────────────────────────────────────────
    model.train()
    running_loss, running_correct, total_samples = 0.0, 0, 0
    # Track epsilon stats across batches for epoch-level monitoring
    epoch_eps_mean, epoch_sigma_mean = [], []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)

        # ── 1. Clean pass ──────────────────────────────────────
        optimizer.zero_grad()
        logits_clean = model(inputs)              # [batch, 26]
        loss_clean   = loss_fn(logits_clean, labels)

        # ── 2. ADES perturbation ───────────────────────────────
        # Replaces: qni_ccp_apcs_perturbation(..., epsilon_matrix)
        # Now each sample gets its own epsilon based on:
        # gradient norm + inverted entropy + inverted MC variance
        perturbed_features, sigma_stats = qni_ccp_ades_perturbation(
            model, inputs, labels, centroids,
            epsilon_min = EPSILON_MIN,
            lambda_max  = LAMBDA_MAX,
            T           = MC_T
        )
        # perturbed_features: [batch, 10] — each row perturbed differently

        # ── 3. QNI pass on perturbed features ─────────────────
        model.train()   # Ensure train mode after MC Dropout eval passes
        q_out_p    = model.q_layer(perturbed_features)   # [batch, 10]
        logits_qni = model.classifier(q_out_p)           # [batch, 26]
        loss_qni   = loss_fn(logits_qni, labels)

        # ── 4. Centroid regularization — identical to APCS ────
        clean_feats         = torch.tanh(model.feature_extractor(inputs))
        current_centroids   = centroids[labels]
        loss_centroid       = F.mse_loss(clean_feats, current_centroids)

        # ── 5. Combined loss — same weights as APCS ───────────
        loss = 0.55 * loss_clean + 0.3 * loss_qni + 0.15 * loss_centroid

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss    += loss.item()
        _, predicted     = torch.max(logits_clean, 1)
        running_correct += (predicted == labels).sum().item()
        total_samples   += labels.size(0)

        # Accumulate epsilon stats for epoch summary
        epoch_eps_mean.append(sigma_stats["eps_mean"])
        epoch_sigma_mean.append(sigma_stats["sigma_mean"])

        loop.set_postfix(
            loss = loss.item(),
            eps  = f"{sigma_stats['eps_mean']:.3f}"
        )

    # ── Epoch summary ─────────────────────────────────────────
    train_loss = running_loss    / len(train_loader)
    train_acc  = running_correct / total_samples
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
    scheduler.step(val_acc)   # Track val_acc (fixed from APCS training)

    # Epoch-level ADES stats — monitor epsilon distribution health
    avg_eps   = np.mean(epoch_eps_mean)
    avg_sigma = np.mean(epoch_sigma_mean)

    print(f"Epoch {epoch+1}/{num_epochs} | {loss_mode}")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    print(f"   ADES | avg_epsilon={avg_eps:.4f} | avg_sigma={avg_sigma:.4f}")

    # ── Checkpoint — track val_acc (not val_loss) ──────────────
    if val_acc > best_val_acc:
        best_val_acc               = val_acc
        epochs_without_improvement = 0
        torch.save(model.state_dict(), ADES_SAVE_PATH)
        print(f"   Best model saved → {ADES_SAVE_PATH}")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= early_stopping_patience:
            print(f"   Early stopping at epoch {epoch+1}")
            break

print(f"\nADES Training complete. Best val_acc: {best_val_acc:.4f}")
print(f"Model saved: {ADES_SAVE_PATH}")

Using device: cuda
Train: 9947 | Val: 2149 | Test: 2130
Class weights computed.

Initializing ADES QNI-CCP Training — Malevis...
  Loaded: /home/netsec1/Kathan/MaleVis experiments/exp-4_1_3.pth
Computing class centroids...

Starting ADES QNI-CCP Training for 70 epochs...
  epsilon_min=0.0164 | lambda_max=3 | MC_T=10
  Per-sample epsilon range: [0.0164, 3.0164]


Epoch 1/70:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 1/70 | Stage 2: Focal (Gamma=1) + ADES
   Train Loss: 0.6644 | Train Acc: 0.8766 | Val Acc: 0.8325
   ADES | avg_epsilon=1.6245 | avg_sigma=0.5360
   Best model saved → qni_ccp_malevis2.pth


Epoch 2/70:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 2/70 | Stage 2: Focal (Gamma=1) + ADES
   Train Loss: 0.5759 | Train Acc: 0.8866 | Val Acc: 0.8018
   ADES | avg_epsilon=1.6326 | avg_sigma=0.5387


...

Epoch 51/70:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 51/70 | Stage 2: Focal (Gamma=1) + ADES
   Train Loss: 0.1151 | Train Acc: 0.9662 | Val Acc: 0.9009
   ADES | avg_epsilon=1.7660 | avg_sigma=0.5832
   Best model saved → qni_ccp_malevis2.pth


Epoch 52/70:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 52/70 | Stage 2: Focal (Gamma=1) + ADES
   Train Loss: 0.1169 | Train Acc: 0.9654 | Val Acc: 0.8948
   ADES | avg_epsilon=1.7676 | avg_sigma=0.5837


Epoch 53/70:   0%|          | 0/622 [00:00<?, ?it/s]